# 01 — Data Overview

Raw data inspection — no modifications, read only.
At the end, write a decision note on which columns to keep.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from Source.scripts.load_data import DATASET_PATHS, load_named_dataset

print('Project root:', PROJECT_ROOT)
print('Available datasets:', list(DATASET_PATHS.keys()))

Project root: C:\Users\hadi\Downloads\TeamsDownloads\Data Analiytcs\Data-Avengers-DA-60
Available datasets: ['the_movies_metadata', 'the_movies_credits', 'tmdb_movies', 'tmdb_credits', 'rt_movies', 'rt_reviews', 'imdb_basics', 'imdb_ratings']


## 1. movies_metadata

Primary source — budget, revenue, genre, release_date and imdb_id are here.

In [2]:
meta = load_named_dataset('the_movies_metadata')
print('Shape:', meta.shape)
meta.head(3)

Shape: (45466, 24)


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0


In [3]:
meta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

In [4]:
meta['budget'] = pd.to_numeric(meta['budget'], errors='coerce')
meta['revenue'] = pd.to_numeric(meta['revenue'], errors='coerce')

usable = ((meta['budget'] > 0) & (meta['revenue'] > 0)).sum()
print(f'Rows with budget > 0 AND revenue > 0: {usable} / {len(meta)}')
print(f'Budget nulls : {meta["budget"].isna().sum()}')
print(f'Revenue nulls: {meta["revenue"].isna().sum()}')
meta[['budget', 'revenue', 'vote_average', 'popularity']].describe()

Rows with budget > 0 AND revenue > 0: 5381 / 45466
Budget nulls : 3
Revenue nulls: 6


,budget,revenue,vote_average
count,4.546300e+04,4.546000e+04,45460.000000
mean,4.224579e+06,1.120935e+07,5.618207
std,1.742413e+07,6.433225e+07,1.924216
min,0.000000e+00,0.000000e+00,0.000000
25%,0.000000e+00,0.000000e+00,5.000000
50%,0.000000e+00,0.000000e+00,6.000000
75%,0.000000e+00,0.000000e+00,6.800000
max,3.800000e+08,2.787965e+09,10.000000


In [5]:
# genres column is a JSON string — inspect its format
meta['genres'].dropna().head(5).tolist()

["[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]",
 "[{'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 10751, 'name': 'Family'}]",
 "[{'id': 10749, 'name': 'Romance'}, {'id': 35, 'name': 'Comedy'}]",
 "[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'name': 'Drama'}, {'id': 10749, 'name': 'Romance'}]",
 "[{'id': 35, 'name': 'Comedy'}]"]

## 2. tmdb_5000_movies

TMDB's own dataset — for popularity and additional budget/revenue validation.

In [6]:
tmdb = load_named_dataset('tmdb_movies')
print('Shape:', tmdb.shape)
tmdb.head(3)

Shape: (4803, 20)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466


In [7]:
usable_tmdb = ((tmdb['budget'] > 0) & (tmdb['revenue'] > 0)).sum()
print(f'Rows with budget > 0 AND revenue > 0: {usable_tmdb} / {len(tmdb)}')
tmdb[['budget', 'revenue', 'vote_average', 'popularity']].describe()

Rows with budget > 0 AND revenue > 0: 3229 / 4803


,budget,revenue,vote_average,popularity
count,4.803000e+03,4.803000e+03,4803.000000,4803.000000
mean,2.904504e+07,8.226064e+07,6.092172,21.492301
std,4.072239e+07,1.628571e+08,1.194612,31.816650
min,0.000000e+00,0.000000e+00,0.000000,0.000000
25%,7.900000e+05,0.000000e+00,5.600000,4.668070
50%,1.500000e+07,1.917000e+07,6.200000,12.921594
75%,4.000000e+07,9.291719e+07,6.800000,28.313505
max,3.800000e+08,2.787965e+09,10.000000,875.581305


## 3. Rotten Tomatoes Movies

For tomatometer (critic score) and audience rating.

In [8]:
rt = load_named_dataset('rt_movies')
print('Shape:', rt.shape)
rt.head(3)

Shape: (17712, 22)


,rotten_tomatoes_link,movie_title,movie_info,critics_consensus,content_rating,genres,directors,authors,actors,original_release_date,...,production_company,tomatometer_status,tomatometer_rating,tomatometer_count,audience_status,audience_rating,audience_count,tomatometer_top_critics_count,tomatometer_fresh_critics_count,tomatometer_rotten_critics_count
0,m/0814255,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",Though it may seem like just another Harry Pot...,PG,"Action & Adventure, Comedy, Drama, Science Fic...",Chris Columbus,"Craig Titley, Chris Columbus, Rick Riordan","Logan Lerman, Brandon T. Jackson, Alexandra Da...",2010-02-12,...,20th Century Fox,Rotten,49.0,149.0,Spilled,53.0,254421.0,43,73,76
1,m/0878835,Please Give,Kate (Catherine Keener) and her husband Alex (...,Nicole Holofcener's newest might seem slight i...,R,Comedy,Nicole Holofcener,Nicole Holofcener,"Catherine Keener, Amanda Peet, Oliver Platt, R...",2010-04-30,...,Sony Pictures Classics,Certified-Fresh,87.0,142.0,Upright,64.0,11574.0,44,123,19
2,m/10,10,"A successful, middle-aged Hollywood songwriter...",Blake Edwards' bawdy comedy may not score a pe...,R,"Comedy, Romance",Blake Edwards,Blake Edwards,"Dudley Moore, Bo Derek, Julie Andrews, Robert ...",1979-10-05,...,Waner Bros.,Fresh,67.0,24.0,Spilled,53.0,14684.0,2,16,8


In [9]:
rating_cols = [c for c in rt.columns if 'rating' in c.lower() or 'score' in c.lower()]
print('Rating-related columns:', rating_cols)
print('\nNull counts:')
print(rt[rating_cols].isna().sum())
print('\nTitle column:', [c for c in rt.columns if 'title' in c.lower()])
rt[rating_cols].describe()

Rating-related columns: ['content_rating', 'tomatometer_rating', 'audience_rating']

Null counts:
content_rating          0
tomatometer_rating     44
audience_rating       296
dtype: int64

Title column: ['movie_title']


,tomatometer_rating,audience_rating
count,17668.000000,17416.000000
mean,60.884763,60.554260
std,28.443348,20.543369
min,0.000000,0.000000
25%,38.000000,45.000000
50%,67.000000,63.000000
75%,86.000000,78.000000
max,100.000000,100.000000


## 4. IMDb Ratings

Most reliable rating source — averageRating and numVotes.
Will be joined via imdb_id, which is much more reliable than title matching.

In [10]:
imdb = load_named_dataset('imdb_ratings')
print('Shape:', imdb.shape)
imdb.head(5)

Shape: (1666284, 3)


,tconst,averageRating,numVotes
0,tt0000001,5.7,2211
1,tt0000002,5.5,317
2,tt0000003,6.4,2325
3,tt0000004,5.1,199
4,tt0000005,6.2,3046


In [11]:
imdb[['averageRating', 'numVotes']].describe()

,averageRating,numVotes
count,1.666284e+06,1.666284e+06
mean,6.960154e+00,1.037396e+03
std,1.419533e+00,1.818265e+04
min,1.000000e+00,5.000000e+00
25%,6.200000e+00,1.200000e+01
50%,7.200000e+00,2.700000e+01
75%,7.900000e+00,1.030000e+02
max,1.000000e+01,3.183071e+06


In [12]:
# Does movies_metadata have imdb_id? Critical for the join.
print('imdb_id in movies_metadata:', 'imdb_id' in meta.columns)
print('Sample imdb_id values:', meta['imdb_id'].dropna().head(5).tolist())
print('Sample tconst values :', imdb['tconst'].head(5).tolist())

imdb_id in movies_metadata: True
Sample imdb_id values: ['tt0114709', 'tt0113497', 'tt0113228', 'tt0114885', 'tt0113041']
Sample tconst values : ['tt0000001', 'tt0000002', 'tt0000003', 'tt0000004', 'tt0000005']


## Decision Note

Copied to `Documentation/reports/data_overview.md`.

### Columns to use

| Source | Columns | Purpose |
|---|---|---|
| movies_metadata | budget, revenue, title, imdb_id, genres, release_date, runtime, vote_average | Primary source |
| tmdb_movies | popularity | Additional feature (as tmdb_popularity) |
| imdb_ratings | averageRating, numVotes | Rating source (joined via imdb_id) |
| rt_movies | tomatometer_rating, audience_rating | Secondary rating source (joined via movie_title) |

### Observations

- movies_metadata total rows: 45,466 — usable (budget > 0 AND revenue > 0): **5,381**
- tmdb_movies total rows: 4,803 — usable (budget > 0 AND revenue > 0): **3,229**
- IMDb ratings total rows: 1,666,284
- RT movies total rows: 17,712
- imdb_id format match: **Yes** — both use `tt` prefix (e.g. `tt0114709` vs `tt0000001`)
- RT tomatometer_rating null rate: **0.2%** (very complete)
- RT audience_rating null rate: **1.7%**
- RT title column name: `movie_title` (not `title`) — handled in merge step
- Primary merge base: movies_metadata (5,381 rows after filter)